# Delta Load Processing (Silver & Gold)\nDetect inserts/updates/deletes/no-change with record hash, MERGE into Delta, manage SCD2 history, and generate NetSuite delta exports.

In [ ]:
from clinical_lakehouse.transformations.delta_load_processing import (\n    detect_delta_changes,\n    merge_delta_current,\n    merge_scd2_history,\n    generate_netsuite_delta_export,\n    default_batch_id,\n)

In [ ]:
batch_id = default_batch_id()\n\n# Example: Silver sponsor incremental load\nsrc_sponsor = spark.table("clinical_migration_dev.silver_stage.silver_sponsor_latest")\ntgt_sponsor = spark.table("clinical_migration_dev.silver.silver_sponsor")\nsponsor_delta = detect_delta_changes(src_sponsor, tgt_sponsor, key_cols=["business_key"])\nmerge_delta_current(spark, "clinical_migration_dev.silver.silver_sponsor", sponsor_delta, key_cols=["business_key"])\nmerge_scd2_history(spark, "clinical_migration_dev.silver_history.silver_sponsor_history", sponsor_delta, business_key_col="business_key")

In [ ]:
# Silver study and contract SCD2\nsrc_study = spark.table("clinical_migration_dev.silver_stage.silver_study_latest")\ntgt_study = spark.table("clinical_migration_dev.silver.silver_study")\nstudy_delta = detect_delta_changes(src_study, tgt_study, key_cols=["business_key"])\nmerge_delta_current(spark, "clinical_migration_dev.silver.silver_study", study_delta, key_cols=["business_key"])\nmerge_scd2_history(spark, "clinical_migration_dev.silver_history.silver_study_history", study_delta, business_key_col="business_key")\n\nsrc_contract = spark.table("clinical_migration_dev.silver_stage.silver_contract_latest")\ntgt_contract = spark.table("clinical_migration_dev.silver.silver_contract")\ncontract_delta = detect_delta_changes(src_contract, tgt_contract, key_cols=["business_key"])\nmerge_delta_current(spark, "clinical_migration_dev.silver.silver_contract", contract_delta, key_cols=["business_key"])\nmerge_scd2_history(spark, "clinical_migration_dev.silver_history.silver_contract_history", contract_delta, business_key_col="business_key")

In [ ]:
# Gold delta detection\ngold_customer_src = spark.table("clinical_migration_dev.gold_stage.gold_netsuite_customer_master_latest")\ngold_customer_tgt = spark.table("clinical_migration_dev.gold.gold_netsuite_customer_master")\ncustomer_delta = detect_delta_changes(gold_customer_src, gold_customer_tgt, key_cols=["customer_external_id"])\nmerge_delta_current(spark, "clinical_migration_dev.gold.gold_netsuite_customer_master", customer_delta, key_cols=["customer_external_id"])\n\ngold_project_src = spark.table("clinical_migration_dev.gold_stage.gold_netsuite_project_master_latest")\ngold_project_tgt = spark.table("clinical_migration_dev.gold.gold_netsuite_project_master")\nproject_delta = detect_delta_changes(gold_project_src, gold_project_tgt, key_cols=["project_external_id"])\nmerge_delta_current(spark, "clinical_migration_dev.gold.gold_netsuite_project_master", project_delta, key_cols=["project_external_id"])\n\ngold_contract_src = spark.table("clinical_migration_dev.gold_stage.gold_netsuite_contract_master_latest")\ngold_contract_tgt = spark.table("clinical_migration_dev.gold.gold_netsuite_contract_master")\ncontract_gold_delta = detect_delta_changes(gold_contract_src, gold_contract_tgt, key_cols=["contract_external_id"])\nmerge_delta_current(spark, "clinical_migration_dev.gold.gold_netsuite_contract_master", contract_gold_delta, key_cols=["contract_external_id"])\n\ngold_invoice_src = spark.table("clinical_migration_dev.gold_stage.gold_netsuite_invoice_header_latest")\ngold_invoice_tgt = spark.table("clinical_migration_dev.gold.gold_netsuite_invoice_header")\ninvoice_delta = detect_delta_changes(gold_invoice_src, gold_invoice_tgt, key_cols=["invoice_external_id"])\nmerge_delta_current(spark, "clinical_migration_dev.gold.gold_netsuite_invoice_header", invoice_delta, key_cols=["invoice_external_id"])\n\ngold_ar_src = spark.table("clinical_migration_dev.gold_stage.gold_netsuite_open_ar_latest")\ngold_ar_tgt = spark.table("clinical_migration_dev.gold.gold_netsuite_open_ar")\nopen_ar_delta = detect_delta_changes(gold_ar_src, gold_ar_tgt, key_cols=["invoice_external_id"])\nmerge_delta_current(spark, "clinical_migration_dev.gold.gold_netsuite_open_ar", open_ar_delta, key_cols=["invoice_external_id"])

In [ ]:
# NetSuite delta exports by batch\nbase_export_path = "exports/netsuite"\ngenerate_netsuite_delta_export(customer_delta, base_export_path, "customer", batch_id)\ngenerate_netsuite_delta_export(project_delta, base_export_path, "project", batch_id)\ngenerate_netsuite_delta_export(contract_gold_delta, base_export_path, "contract", batch_id)\ngenerate_netsuite_delta_export(invoice_delta, base_export_path, "invoice", batch_id)\ngenerate_netsuite_delta_export(open_ar_delta, base_export_path, "open_ar", batch_id)